In [42]:
import sys
sys.path.append("..")
from src.model.geoclip import GeoCLIP
from src.model.g3 import G3
from src.utils import build_index, add_record_to_index, save_index, get_device, read_index
import polars as pl
import torch
from torch.nn import functional as F
from tqdm import tqdm

In [2]:
DEVICE = get_device()

In [3]:
coords_df = pl.read_csv("../src/model/geoclip/gps_gallery/coordinates_100K_geo.csv")
coords_df.head()

LAT,LON,country_code,country,region,subregion,city
f64,f64,str,str,str,str,str
-50.943392,-72.935664,"""AR""","""Argentina""","""South America""","""South America""","""Rio Turbio"""
34.201047,-118.599931,"""US""","""United States""","""North America""","""Northern America""","""Canoga Park"""
48.839963,-3.504874,"""FR""","""France""","""Europe""","""Western Europe""","""Tregastel"""
38.942855,-119.975852,"""US""","""United States""","""North America""","""Northern America""","""South Lake Tahoe"""
19.412728,-99.168949,"""MX""","""Mexico""","""North America""","""Central America""","""Benito Juarez"""


In [4]:
geoclip = GeoCLIP().to(DEVICE).eval()
g3 = G3.from_pretrained(device=DEVICE).eval()

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
geoclip_index = build_index(d=512)
g3_index = build_index(d=512)

In [35]:
gps_tensor = torch.Tensor(coords_df.select(["LAT", "LON"]).to_numpy()).to(DEVICE)

with torch.no_grad():
    for gps in tqdm(gps_tensor, desc="embed gps"):
        # geoclip_embed = F.normalize(geoclip.location_encoder(gps.view(1, -1)), dim=-1).cpu().numpy()
        # add_record_to_index(geoclip_index, geoclip_embed)

        # g3_embed = F.normalize(g3.loc2img_proj(g3.location_encoder(gps.view(1, -1))), dim=-1).cpu().numpy()
        g3_embed = F.normalize(g3.location_encoder(gps.view(1, -1)), dim=-1).cpu().numpy()
        add_record_to_index(g3_index, g3_embed)

embed gps:   0%|          | 0/100000 [00:00<?, ?it/s]

embed gps: 100%|██████████| 100000/100000 [07:45<00:00, 215.04it/s]


In [36]:
# save_index(geoclip_index, coords_df.to_dicts(), "../index/geoclip-gps-gallery")
save_index(g3_index, coords_df.to_dicts(), "../index/g3-wo-proj-gps-gallery")

In [43]:
g3_index_old, meta = read_index("../index/g3-gps-gallery")

In [49]:
query = "a tower in tokyo"
# query_emb = geoclip.text_encoder(query)
with torch.no_grad():
    # query_tok = geoclip.text_encoder.preprocess_text(query)
    # query_emb = geoclip.text_encoder(**{k:v.to(DEVICE) for k, v in query_tok.items()}).cpu().numpy()
    query_tok = g3.preprocess_text(query)
    query_emb = g3.text_proj(g3.text_model(**{k:v.to(DEVICE) for k, v in query_tok.items()}).pooler_output)
    query_emb = F.normalize(g3.txt2img_proj(query_emb.reshape(query_emb.shape[0], -1)), dim=-1).cpu().numpy()
    # query_emb = F.normalize(g3.text_model(**{k:v.to(DEVICE) for k, v in query_tok.items()}).pooler_output, dim=-1).cpu().numpy()

In [52]:
D, I = g3_index_old.search(query_emb, 100)

In [53]:
meta = coords_df.to_dicts()
rdocs = [meta[i] for i in I.flatten().tolist()]

In [54]:
rdocs

[{'LAT': 40.7635,
  'LON': -73.966001,
  'country_code': 'US',
  'country': 'United States',
  'region': 'North America',
  'subregion': 'Northern America',
  'city': 'Manhattan'},
 {'LAT': 40.761508,
  'LON': -73.967567,
  'country_code': 'US',
  'country': 'United States',
  'region': 'North America',
  'subregion': 'Northern America',
  'city': 'Manhattan'},
 {'LAT': 40.758238,
  'LON': -73.966441,
  'country_code': 'US',
  'country': 'United States',
  'region': 'North America',
  'subregion': 'Northern America',
  'city': 'Long Island City'},
 {'LAT': 40.7625,
  'LON': -73.967,
  'country_code': 'US',
  'country': 'United States',
  'region': 'North America',
  'subregion': 'Northern America',
  'city': 'Manhattan'},
 {'LAT': 40.758806,
  'LON': -73.970496,
  'country_code': 'US',
  'country': 'United States',
  'region': 'North America',
  'subregion': 'Northern America',
  'city': 'Manhattan'},
 {'LAT': 40.756183,
  'LON': -73.969566,
  'country_code': 'US',
  'country': 'United